# Pipis 6.7 Ultra Hammamovich — QLoRA experiment

Экспериментальное дообучение Qwen2.5-1.5B-Instruct через QLoRA на Google Colab T4. Запускай сверху вниз. Начни с небольшого числа шагов; это тестовый прототип, а не готовая модель.

## 1. Проверка GPU

Убедись, что подключена Tesla T4 и доступна CUDA.

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не подключён. В Colab выбери Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Установка совместимых библиотек

Эта ячейка может занять несколько минут. Если Colab попросит перезапустить среду, перезапусти её и затем запускай ячейки сверху снова.

In [ ]:
%pip install -q transformers==4.48.3 datasets==3.2.0 peft==0.14.0 trl==0.14.0 accelerate==1.3.0 bitsandbytes==0.45.2 sentencepiece


In [ ]:
import sys, inspect, torch, transformers, datasets, peft, trl, accelerate, bitsandbytes
from trl import SFTTrainer, SFTConfig
print('Python:', sys.version.split()[0])
for package in (torch, transformers, datasets, peft, trl, accelerate, bitsandbytes):
    print(f'{package.__name__}: {package.__version__}')
print('SFTTrainer processing_class supported:', 'processing_class' in inspect.signature(SFTTrainer.__init__).parameters)
sft_params = inspect.signature(SFTConfig.__init__).parameters
print('SFTConfig max_seq_length supported:', 'max_seq_length' in sft_params)
print('SFTConfig max_length supported:', 'max_length' in sft_params)
print('SFTConfig eval_strategy supported:', 'eval_strategy' in sft_params)
print('SFTConfig evaluation_strategy supported:', 'evaluation_strategy' in sft_params)
if 'processing_class' not in inspect.signature(SFTTrainer.__init__).parameters:
    raise RuntimeError('Установленная TRL не поддерживает ожидаемый processing_class.')
if not ({'max_seq_length', 'max_length'} & set(sft_params)):
    raise RuntimeError('Установленная TRL не поддерживает max_seq_length или max_length; не запускай обучение, сохрани вывод версий.')


## 3. Загрузить данные из GitHub

Здесь используется только тренировочный файл. Проверочный набор не участвует в обучении.

In [ ]:
from datasets import load_dataset
train_url = 'https://raw.githubusercontent.com/shadowfightdimaonly-del/Pipis-6.7/main/data/pipis_train.jsonl'
eval_url = 'https://raw.githubusercontent.com/shadowfightdimaonly-del/Pipis-6.7/main/data/pipis_eval.jsonl'
train_dataset = load_dataset('json', data_files=train_url, split='train')
eval_dataset = load_dataset('json', data_files=eval_url, split='train')
print('Train rows:', len(train_dataset))
print('Eval rows:', len(eval_dataset))
print(train_dataset[0]['text'])
assert len(train_dataset) > 0 and len(eval_dataset) > 0
assert 'text' in train_dataset.column_names and 'text' in eval_dataset.column_names

def validate_text_split(dataset, split_name):
    texts = []
    for index, row in enumerate(dataset):
        text = row.get('text')
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f'{split_name}[{index}] must contain non-empty string field "text"')
        texts.append(text.strip())
    duplicates = len(texts) - len(set(texts))
    if duplicates:
        raise ValueError(f'{split_name} contains {duplicates} duplicate text(s)')
    return set(texts)

train_texts = validate_text_split(train_dataset, 'train')
eval_texts = validate_text_split(eval_dataset, 'eval')
overlap = train_texts & eval_texts
if overlap:
    raise ValueError(f'{len(overlap)} exact text(s) appear in both train and eval')
print('Dataset validation passed: non-empty text fields, no exact duplicates, no train/eval overlap.')

## 4. Загрузить базовую модель в 4-битном режиме

Модель: Qwen/Qwen2.5-1.5B-Instruct. Необходим интернет-доступ в Colab. Первый запуск скачает веса и может занять время.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print('Модель загружена в 4-битном режиме.')

## 5. Настроить и запустить короткий пробный запуск

Набор сейчас маленький и предназначен только для проверки конвейера. Если тренировка завершится, это ещё не доказывает, что качество модели улучшилось. Теперь проверочная выборка используется для оценки loss, но это ещё не полноценная оценка качества ответов.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

sft_config_kwargs = dict(
    output_dir='/content/pipis-6.7-qlora-output',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_steps=10,
    logging_steps=1,
    eval_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    dataset_text_field='text',
    packing=False,
)

# TRL renamed these arguments across versions; choose supported names dynamically.
sft_params = inspect.signature(SFTConfig.__init__).parameters
if 'eval_strategy' in sft_params:
    sft_config_kwargs['eval_strategy'] = 'steps'
elif 'evaluation_strategy' in sft_params:
    sft_config_kwargs['evaluation_strategy'] = 'steps'
else:
    raise RuntimeError('SFTConfig не поддерживает настройку evaluation strategy.')
if 'max_seq_length' in sft_params:
    sft_config_kwargs['max_seq_length'] = 512
elif 'max_length' in sft_params:
    sft_config_kwargs['max_length'] = 512
else:
    raise RuntimeError('SFTConfig не поддерживает ограничение длины последовательности.')
training_args = SFTConfig(**sft_config_kwargs)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.train()
eval_results = trainer.evaluate()
print('Eval results:', eval_results)

## 6. Сохранить адаптер

Сохраняем только обучаемый LoRA-адаптер, а не полную базовую модель. Затем можно скачать папку из Colab или сохранить её в Google Drive.

In [ ]:
adapter_dir = '/content/pipis-6.7-adapter'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Адаптер сохранён:', adapter_dir)


## 7. Проверить, что адаптер загружается обратно

Эта ячейка повторно загружает базовую модель и подключает сохранённый адаптер. Это требует дополнительной загрузки весов и времени.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

reload_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
reload_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=reload_quant_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
reloaded_model = PeftModel.from_pretrained(reload_base, adapter_dir)
reloaded_model.eval()
test_prompt = 'Пользователь: Кто ты?\nПипис:'
inputs = tokenizer(test_prompt, return_tensors='pt').to(reloaded_model.device)
with torch.no_grad():
    generated = reloaded_model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated[0], skip_special_tokens=True))
print('Проверка завершена: адаптер удалось подключить и модель сгенерировала ответ.')


## 8. Сравнить базовую модель и адаптер на одинаковых запросах

Это быстрый качественный тест, а не строгий бенчмарк. Для каждого запроса выводятся ответы без адаптера и с адаптером.

In [ ]:
comparison_prompts = [
    'Пользователь: Кто ты?\nПипис:',
    'Пользователь: Объясни LoRA простыми словами.\nПипис:',
    'Пользователь: Что ты сделаешь, если не уверен в ответе?\nПипис:',
]

def generate_answer(active_model, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(active_model.device)
    with torch.no_grad():
        output = active_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

for prompt in comparison_prompts:
    print('\n' + '=' * 70)
    print(prompt)
    with trainer.model.disable_adapter():
        base_answer = generate_answer(trainer.model, prompt)
    tuned_answer = generate_answer(trainer.model, prompt)
    print('\n[BASE MODEL]\n', base_answer)
    print('\n[PIPIS ADAPTER]\n', tuned_answer)


## 9. Упаковать адаптер для скачивания

Скачивается только адаптер и токенизатор, не базовая модель. Для использования адаптера базовая модель понадобится отдельно.

In [ ]:
from google.colab import files
!cd /content && zip -qr pipis-6.7-adapter.zip pipis-6.7-adapter
files.download('/content/pipis-6.7-adapter.zip')


## Важные ограничения

- 30 тренировочных примеров — лишь проверочный черновик. Для заметного улучшения нужно расширить и вручную проверить датасет.
- Не добавляй данные из eval в train.
- Ячейка повторной загрузки дополнительно скачивает базовую модель и может занять время/VRAM.
- Если ячейка падает из-за версии библиотеки или нехватки памяти, сохрани полный текст ошибки и остановись; не меняй случайные параметры.
- Не загружай секреты или личные данные в публичный репозиторий.
- Сравнение ответов — ручная проверка на нескольких примерах, не доказательство общего улучшения.
- Адаптер — не самостоятельная полная модель. Для запуска понадобится базовая модель и этот адаптер.
